# Sequence Augmentation Ablation

Ce notebook reprend le script `sequence_augmentation_ablation.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Teste les augmentations temporelles pour rendre le modele sequence live plus robuste.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Repeated-split ablation for sequence augmentation policies.
- Commande de reproduction referencee : sequence augmentation ablation.
- Artefacts controles : Repeated-split sequence augmentation ablation exists. (`runs/exp_044_sequence_augmentation_ablation/metrics/sequence_augmentation_ablation_summary.csv`).
- Run par defaut : `runs/exp_044_sequence_augmentation_ablation`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "sequence_augmentation_ablation.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import time
from dataclasses import dataclass
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score
from torch.utils.data import DataLoader, Dataset

from ml_pipeline import HORIZONS, ROOT, safe_auc, write_json
from sequence_experiments import (
    evaluate_catalogue_model,
    make_loss,
    make_model,
    make_run_dir,
    predict_model,
    resample_sequence_np,
    set_seed,
)
from sequence_stability_experiments import normalize_for_split, stratified_video_split


DEFAULT_SEEDS = [
    101,
    202,
    303,
    404,
    505,
    606,
    707,
    808,
    909,
    1001,
    1101,
    1202,
    1303,
    1404,
    1505,
    1606,
    1707,
    1808,
    1909,
    2001,
]


## Classe `AugPolicy`

Cette cellule definit `AugPolicy`. Elle prepare une partie du script.

In [ ]:
@dataclass(frozen=True)
class AugPolicy:
    name: str
    noise_prob: float = 0.0
    noise_std: float = 0.0
    dropout_prob: float = 0.0
    dropout_min_frac: float = 0.03
    dropout_max_frac: float = 0.10
    cutout_prob: float = 0.0
    cutout_max_frac: float = 0.25
    speed_prob: float = 0.0
    speed_min: float = 0.85
    speed_max: float = 1.18
    mixup_alpha: float = 0.0
    mixup_prob: float = 0.0


POLICIES = [
    AugPolicy("none"),
    AugPolicy("jitter_only", noise_prob=0.85, noise_std=0.035),
    AugPolicy("channel_dropout_only", dropout_prob=0.35),
    AugPolicy("temporal_cutout_only", cutout_prob=0.35),
    AugPolicy("speed_resample_only", speed_prob=0.35),
    AugPolicy("mild_all", noise_prob=0.60, noise_std=0.020, dropout_prob=0.20, cutout_prob=0.20, cutout_max_frac=0.18, speed_prob=0.20, speed_min=0.92, speed_max=1.10),
    AugPolicy("standard_all", noise_prob=0.85, noise_std=0.035, dropout_prob=0.35, cutout_prob=0.35, cutout_max_frac=0.25, speed_prob=0.35, speed_min=0.85, speed_max=1.18),
    AugPolicy("strong_all", noise_prob=0.95, noise_std=0.055, dropout_prob=0.55, dropout_min_frac=0.05, dropout_max_frac=0.15, cutout_prob=0.55, cutout_max_frac=0.35, speed_prob=0.55, speed_min=0.75, speed_max=1.30),
    AugPolicy("standard_all_mixup", noise_prob=0.85, noise_std=0.035, dropout_prob=0.35, cutout_prob=0.35, cutout_max_frac=0.25, speed_prob=0.35, speed_min=0.85, speed_max=1.18, mixup_alpha=0.20, mixup_prob=0.50),
]


## Classe `PolicySequenceDataset`

Cette cellule definit `PolicySequenceDataset`. Elle prepare une partie du script.

In [ ]:
class PolicySequenceDataset(Dataset):
    def __init__(self, X, y, indices, policy, seed=42):
        self.X = X
        self.y = y
        self.indices = np.asarray(indices, dtype=np.int64)
        self.policy = policy
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        source_idx = self.indices[idx]
        x = self.X[source_idx].copy()
        y = self.y[source_idx].copy()
        x = self.apply_policy(x)
        return torch.from_numpy(x.astype(np.float32)), torch.from_numpy(y.astype(np.float32))

    def apply_policy(self, x):
        p = self.policy
        if p.noise_prob > 0 and self.rng.random() < p.noise_prob:
            x += self.rng.normal(0.0, p.noise_std, size=x.shape).astype(np.float32)
        if p.dropout_prob > 0 and self.rng.random() < p.dropout_prob:
            frac = float(self.rng.uniform(p.dropout_min_frac, p.dropout_max_frac))
            n_features = max(1, int(x.shape[1] * frac))
            cols = self.rng.choice(x.shape[1], size=n_features, replace=False)
            x[:, cols] = 0.0
        if p.cutout_prob > 0 and self.rng.random() < p.cutout_prob:
            max_width = max(3, int(round(x.shape[0] * p.cutout_max_frac)))
            width = int(self.rng.integers(2, max_width + 1))
            start = int(self.rng.integers(0, max(1, x.shape[0] - width + 1)))
            x[start : start + width] = 0.0
        if p.speed_prob > 0 and self.rng.random() < p.speed_prob:
            scale = float(self.rng.uniform(p.speed_min, p.speed_max))
            x = resample_sequence_np(x, scale)
        return x


## Classe `PlainSequenceDataset`

Cette cellule definit `PlainSequenceDataset`. Elle prepare une partie du script.

In [ ]:
class PlainSequenceDataset(Dataset):
    def __init__(self, X, y, indices):
        self.X = X
        self.y = y
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        source_idx = self.indices[idx]
        return torch.from_numpy(self.X[source_idx].astype(np.float32)), torch.from_numpy(self.y[source_idx].astype(np.float32))


## Fonction `mixup_batch`

Cette cellule definit `mixup_batch`. Elle prepare une partie du script.

In [ ]:
def mixup_batch(x, y, alpha, prob):
    if alpha <= 0 or prob <= 0 or torch.rand(()) > prob or x.shape[0] < 2:
        return x, y
    lam = torch.distributions.Beta(alpha, alpha).sample().to(x.device)
    perm = torch.randperm(x.shape[0], device=x.device)
    return lam * x + (1.0 - lam) * x[perm], lam * y + (1.0 - lam) * y[perm]


## Fonction `load_raw_sequence`

Cette cellule definit `load_raw_sequence`. Elle prepare une partie du script.

In [ ]:
def load_raw_sequence(sequence_run):
    sequence_run = sequence_run if sequence_run.is_absolute() else ROOT / sequence_run
    data = np.load(sequence_run / "features" / "sequence_dataset.npz")
    X_norm = data["X"].astype(np.float32)
    y = data["y"].astype(np.float32)
    mean = data["mean"].astype(np.float32)
    std = data["std"].astype(np.float32)
    X_raw = X_norm * std.reshape(1, 1, -1) + mean.reshape(1, 1, -1)
    meta = pd.read_csv(sequence_run / "features" / "sequence_index.csv")
    return sequence_run, X_raw.astype(np.float32), y, meta


## Fonction `train_policy_model`

Cette cellule definit `train_policy_model`. Elle prepare une partie du script.

In [ ]:
def train_policy_model(model_name, policy, X, y, meta, run_dir, args, device):
    set_seed(args.seed)
    seq_len, input_dim, out_dim = X.shape[1], X.shape[2], y.shape[1]
    train_idx = np.flatnonzero(meta["split"].to_numpy() == "train")
    val_idx = np.flatnonzero(meta["split"].to_numpy() == "val")
    train_ds = PolicySequenceDataset(X, y, train_idx, policy=policy, seed=args.seed)
    val_ds = PlainSequenceDataset(X, y, val_idx)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, num_workers=0)

    model = make_model(args.architecture, seq_len, input_dim, out_dim).to(device)
    positives = y[train_idx].sum(axis=0)
    negatives = len(train_idx) - positives
    pos_weight = torch.tensor(np.clip(negatives / np.maximum(positives, 1.0), 1.0, 20.0), dtype=torch.float32, device=device)
    loss_args = SimpleNamespace(loss=args.loss, label_smoothing=args.label_smoothing, focal_gamma=args.focal_gamma)
    criterion = make_loss(loss_args, pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

    best = {"score": -1.0, "epoch": 0, "state": None}
    patience_left = args.patience
    history = []
    h1_idx = HORIZONS.index(1.0)
    start = time.perf_counter()
    for epoch in range(1, args.epochs + 1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            xb, yb = mixup_batch(xb, yb, policy.mixup_alpha, policy.mixup_prob)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        val_probs, val_targets = predict_model(model, val_loader, device)
        val_ap = safe_auc(average_precision_score, val_targets[:, h1_idx], val_probs[:, h1_idx])
        val_score = float(val_ap or 0.0)
        scheduler.step(val_score)
        history.append(
            {
                "model": model_name,
                "policy": policy.name,
                "epoch": epoch,
                "train_loss": float(np.mean(losses)),
                "val_ap_h1": val_score,
                "lr": float(optimizer.param_groups[0]["lr"]),
            }
        )
        if val_score > best["score"] + 1e-5:
            best = {"score": val_score, "epoch": epoch, "state": {k: v.detach().cpu() for k, v in model.state_dict().items()}}
            patience_left = args.patience
        else:
            patience_left -= 1
        if patience_left <= 0:
            break

    if best["state"] is not None:
        model.load_state_dict(best["state"])
    train_time_s = time.perf_counter() - start
    model_path = run_dir / "models" / f"{model_name}.pt"
    torch.save(
        {
            "model_name": model_name,
            "architecture": args.architecture,
            "loss": args.loss,
            "policy": policy.__dict__,
            "state_dict": model.state_dict(),
            "seq_len": int(seq_len),
            "input_dim": int(input_dim),
            "horizons": HORIZONS,
            "best_epoch": int(best["epoch"]),
            "best_val_ap_h1": float(best["score"]),
        },
        model_path,
    )
    return model, history, train_time_s, model_path.stat().st_size


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(metrics):
    h1 = metrics[metrics["horizon_s"].astype(float).eq(1.0)].copy()
    rows = []
    for (policy, split), group in h1.groupby(["augmentation_policy", "split"]):
        rows.append(
            {
                "augmentation_policy": policy,
                "split": split,
                "n_repeats": int(group["repeat_seed"].nunique()),
                "ap_mean": float(group["average_precision"].mean()),
                "ap_std": float(group["average_precision"].std(ddof=0)),
                "roc_auc_mean": float(group["roc_auc"].mean()),
                "hit_rate_mean": float(group["best_hit_rate"].mean()),
                "false_alarms_per_min_mean": float(group["best_false_alarms_per_min"].mean()),
                "precision_mean": float(group["best_window_precision"].mean()),
                "inference_ms_per_window_mean": float(group["inference_ms_per_window"].mean()),
            }
        )
    return pd.DataFrame(rows)


## Fonction `write_summary`

Cette cellule definit `write_summary`. Elle prepare une partie du script.

In [ ]:
def write_summary(run_dir, summary, args):
    val = summary[summary["split"].eq("val")].copy()
    val["selection_score"] = val["ap_mean"] + 0.5 * val["hit_rate_mean"] + 0.2 * val["precision_mean"] - 0.03 * val["false_alarms_per_min_mean"].clip(upper=20)
    val = val.sort_values("selection_score", ascending=False)
    test = summary[summary["split"].eq("test")].sort_values("ap_mean", ascending=False)

    lines = ["# Sequence Augmentation Ablation", ""]
    lines.append("This tests which train-split-only sequence augmentation policies help the danger model under repeated parent-video splits.")
    lines.append("")
    lines.append(f"- Architecture: `{args.architecture}`")
    lines.append(f"- Loss: `{args.loss}`")
    lines.append(f"- Repeats: `{len(args.seeds)}`")
    lines.append(f"- Seeds: `{args.seeds}`")
    lines.append("- Normalization: recomputed from train videos inside each repeat.")
    lines.append("")
    lines.append("## Validation Ranking")
    lines.append("")
    lines.append("| rank | policy | AP mean | AP std | hit | FA/min | precision |")
    lines.append("|---:|---|---:|---:|---:|---:|---:|")
    for rank, (_, row) in enumerate(val.iterrows(), start=1):
        lines.append(
            f"| {rank} | {row['augmentation_policy']} | {row['ap_mean']:.3f} | {row['ap_std']:.3f} | "
            f"{row['hit_rate_mean']:.3f} | {row['false_alarms_per_min_mean']:.3f} | {row['precision_mean']:.3f} |"
        )
    lines.append("")
    lines.append("## Test Diagnostics")
    lines.append("")
    lines.append("| rank | policy | AP mean | AP std | ROC AUC | hit | FA/min | precision |")
    lines.append("|---:|---|---:|---:|---:|---:|---:|---:|")
    for rank, (_, row) in enumerate(test.iterrows(), start=1):
        lines.append(
            f"| {rank} | {row['augmentation_policy']} | {row['ap_mean']:.3f} | {row['ap_std']:.3f} | "
            f"{row['roc_auc_mean']:.3f} | {row['hit_rate_mean']:.3f} | {row['false_alarms_per_min_mean']:.3f} | {row['precision_mean']:.3f} |"
        )
    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- This isolates augmentation policy while holding architecture, loss, split strategy, and normalization policy fixed.")
    lines.append("- Validation ranking should drive policy choice; test ranking is diagnostic.")
    lines.append("- Strong augmentation or mixup should only be adopted if it improves repeated-split validation without damaging false-alarm behavior.")
    summary_path = run_dir / "sequence_augmentation_ablation_summary.md"
    summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return summary_path


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    run_dir = make_run_dir(args.run_name)
    source_run, X_raw, y, base_meta = load_raw_sequence(ROOT / args.sequence_run)
    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)
    policies = POLICIES[:2] if args.quick else POLICIES

    write_json(
        run_dir / "metrics" / "sequence_augmentation_ablation_config.json",
        {
            "sequence_run": str(source_run),
            "architecture": args.architecture,
            "loss": args.loss,
            "seeds": args.seeds,
            "policies": [policy.__dict__ for policy in policies],
            "split_policy": "repeated parent-video split; sequence windows inherit split",
            "normalization": "train split mean/std recomputed per repeat",
        },
    )

    all_metrics = []
    all_history = []
    split_rows = []
    for seed in args.seeds:
        split = stratified_video_split(base_meta, seed)
        meta = base_meta.copy()
        meta["split"] = meta["video_id"].map(split)
        X, split_mean, split_std = normalize_for_split(X_raw, meta)
        meta.to_csv(run_dir / "features" / f"split_seed_{seed}.csv", index=False)
        np.savez_compressed(run_dir / "features" / f"normalizer_seed_{seed}.npz", mean=split_mean, std=split_std)
        split_rows.append({"seed": seed, **meta.groupby("split")["video_id"].nunique().to_dict()})

        for policy in policies:
            model_name = f"seed{seed}_{args.architecture}_{args.loss}_{policy.name}"
            print(f"training {model_name}")
            model_args = SimpleNamespace(**vars(args), seed=seed)
            model, history, train_time_s, model_size_bytes = train_policy_model(model_name, policy, X, y, meta, run_dir, model_args, device)
            for row in history:
                row["repeat_seed"] = seed
            all_history.extend(history)
            rows, _ = evaluate_catalogue_model(model_name, model, X, y, meta, run_dir, device, train_time_s, model_size_bytes, args.batch_size, args.loss)
            for row in rows:
                row["repeat_seed"] = seed
                row["augmentation_policy"] = policy.name
            all_metrics.extend(rows)
            pd.DataFrame(all_metrics).to_csv(run_dir / "metrics" / "sequence_augmentation_ablation_metrics.csv", index=False)
            pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "sequence_augmentation_ablation_training_history.csv", index=False)

    metrics = pd.DataFrame(all_metrics)
    metrics.to_csv(run_dir / "metrics" / "sequence_augmentation_ablation_metrics.csv", index=False)
    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "sequence_augmentation_ablation_training_history.csv", index=False)
    pd.DataFrame(split_rows).to_csv(run_dir / "metrics" / "sequence_augmentation_ablation_split_counts.csv", index=False)
    summary = summarize(metrics)
    summary.to_csv(run_dir / "metrics" / "sequence_augmentation_ablation_summary.csv", index=False)
    write_summary(run_dir, summary, args)
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Repeated-split ablation for sequence augmentation policies.")
    parser.add_argument("--sequence-run", default="runs/exp_008_sequence_len60_catalogue")
    parser.add_argument("--run-name", default="exp_044_sequence_augmentation_ablation")
    parser.add_argument("--seeds", nargs="+", type=int, default=DEFAULT_SEEDS)
    parser.add_argument("--architecture", default="tcn", choices=["tcn", "cnn1d", "gru", "lstm", "cnn_gru", "transformer", "flatten_mlp"])
    parser.add_argument("--loss", default="focal", choices=["bce", "focal", "smooth_bce"])
    parser.add_argument("--epochs", type=int, default=18)
    parser.add_argument("--patience", type=int, default=4)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--label-smoothing", type=float, default=0.05)
    parser.add_argument("--focal-gamma", type=float, default=2.0)
    parser.add_argument("--grad-clip", type=float, default=3.0)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--quick", action="store_true")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_044_sequence_augmentation_ablation_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["sequence_augmentation_ablation.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
